# Hibiscus Transformer — Train Small (12M) on Colab

This notebook trains the small (12M parameter) transformer on **TinyStories** using Google Colab Pro.

**Runtime setup:** Go to `Runtime > Change runtime type > GPU (T4 or A100)`

## 1. Setup

In [ ]:
# Clean old clone and re-clone fresh
!rm -rf /content/Hibiscus
!git clone https://github.com/prashanth8983/Hibiscus.git /content/Hibiscus
%cd /content/Hibiscus

In [9]:
# Install dependencies
!pip install -q torch torchvision datasets tokenizers transformers \
    tqdm wandb tensorboard pyyaml matplotlib seaborn scipy scikit-learn

In [10]:
# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


## 2. Load Dataset & Train Tokenizer

In [11]:
import random
import numpy as np
from datasets import load_dataset
from transformer.config import Config
from transformer.tokenizer import Tokenizer

# Load config
config = Config.from_yaml("configs/small.yaml")

# Create dirs if ensure_dirs exists (newer code), otherwise they're auto-created
if hasattr(config, 'ensure_dirs'):
    config.ensure_dirs()

# Reduce epochs for Colab (adjust as needed)
config.training.max_epochs = 10

# Set seeds
random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

print(f"Config loaded: {config.model.d_model}d, {config.model.n_heads}h, {config.model.n_layers}L")

TypeError: '<=' not supported between instances of 'str' and 'int'

In [ ]:
# Load TinyStories dataset
print("Loading TinyStories dataset...")
dataset = load_dataset("roneneldan/TinyStories")
print(f"Train samples: {len(dataset['train']):,}")
print(f"Val samples: {len(dataset['validation']):,}")
print(f"\nSample story:\n{dataset['train'][0]['text'][:300]}...")

In [ ]:
# Train tokenizer on a subset (full dataset takes too long for BPE)
tokenizer = Tokenizer(
    tokenizer_type=config.data.tokenizer_type,
    vocab_size=config.data.vocab_size,
    min_freq=config.data.min_freq,
    pad_token=config.data.pad_token,
    unk_token=config.data.unk_token,
    bos_token=config.data.bos_token,
    eos_token=config.data.eos_token
)

# Use first 50k samples for tokenizer training (faster)
TOKENIZER_TRAIN_SIZE = 50_000
print(f"Training tokenizer on {TOKENIZER_TRAIN_SIZE:,} samples...")

def tokenizer_texts():
    for i, example in enumerate(dataset["train"]):
        if i >= TOKENIZER_TRAIN_SIZE:
            break
        yield example["text"]

tokenizer.train_from_iterator(tokenizer_texts())
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Save tokenizer
import os
os.makedirs("checkpoints", exist_ok=True)
tokenizer.save("checkpoints/tokenizer.pkl")
print("Tokenizer saved.")

## 3. Create DataLoaders

In [ ]:
from transformer.data import HuggingFaceTextDataset, collate_fn
from torch.utils.data import DataLoader

# Use a subset for faster training on Colab
# Adjust TRAIN_SIZE based on how long you want to train
TRAIN_SIZE = 200_000  # ~200k stories
VAL_SIZE = 5_000

train_subset = dataset["train"].select(range(min(TRAIN_SIZE, len(dataset["train"]))))
val_subset = dataset["validation"].select(range(min(VAL_SIZE, len(dataset["validation"]))))

print(f"Creating datasets (train={len(train_subset):,}, val={len(val_subset):,})...")
train_dataset = HuggingFaceTextDataset(train_subset, config.data, tokenizer)
val_dataset = HuggingFaceTextDataset(val_subset, config.data, tokenizer)

print(f"Train pairs: {len(train_dataset):,}")
print(f"Val pairs: {len(val_dataset):,}")

pad_token_id = tokenizer.pad_token_id

train_loader = DataLoader(
    train_dataset,
    batch_size=config.data.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=lambda batch: collate_fn(batch, pad_token_id)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.data.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=lambda batch: collate_fn(batch, pad_token_id)
)

print(f"Train batches: {len(train_loader):,}")
print(f"Val batches: {len(val_loader):,}")

## 4. Create Model

In [ ]:
from transformer.model import Transformer

# Update vocab_size to match actual tokenizer vocab
config.model.vocab_size = tokenizer.vocab_size

model = Transformer(config.model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model parameters: {model.count_parameters():,}")
print(f"Model size: {model.get_model_size()}")
print(f"Device: {device}")

## 5. Train

In [ ]:
from transformer.trainer import Trainer

# Disable W&B for Colab (uses tensorboard instead)
config.use_wandb = False
config.use_tensorboard = True

trainer = Trainer(
    model=model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer
)

print(f"Training for {config.training.max_epochs} epochs...")
print(f"Batches per epoch: {len(train_loader):,}")
print(f"Total steps: {len(train_loader) * config.training.max_epochs:,}")
print("---")

trainer.train()

## 6. Generate Text

In [ ]:
# Generate sample stories
prompts = [
    "Once upon a time",
    "The little girl",
    "There was a big",
    "One day the cat",
    "The boy went to"
]

print("=" * 60)
print("Generated Stories")
print("=" * 60)

for prompt in prompts:
    generated = trainer.generate_text(
        prompt=prompt,
        max_length=100,
        temperature=0.8,
        top_k=50,
        top_p=0.9
    )
    print(f"\nPrompt: '{prompt}'")
    print(f"Output: {generated}")
    print("-" * 40)

## 7. View Training Curves

In [ ]:
# Launch TensorBoard inline
%load_ext tensorboard
%tensorboard --logdir logs

## 8. Save to Google Drive (Optional)

In [ ]:
# Mount Google Drive and copy checkpoint
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = "/content/drive/MyDrive/hibiscus_checkpoints"
os.makedirs(save_dir, exist_ok=True)

# Copy best model and tokenizer
for f in ["checkpoints/best_model.pt", "checkpoints/tokenizer.pkl"]:
    if os.path.exists(f):
        shutil.copy(f, save_dir)
        print(f"Saved {f} -> {save_dir}")

# Also save the config
shutil.copy("configs/small.yaml", save_dir)
print(f"\nAll files saved to {save_dir}")